In [ ]:
!pip install -q datasets torchaudio accelerate soundfile webdataset librosa
!git clone https://github.com/k2-fsa/OmniVoice.git
%cd OmniVoice
!pip install -e .

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 2.9 MB/s eta 0:00:00
Cloning into 'OmniVoice'...
remote: Enumerating objects: 265, done.
remote: Counting objects: 100% (175/175), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 265 (delta 108), reused 75 (delta 75), pack-reused 90 (from 1)
Receiving objects: 100% (265/265), 1.20 MiB | 7.92 MiB/s, done.
Resolving deltas: 100% (129/129), done.
/content/OmniVoice
Obtaining file:///content/OmniVoice
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 8.5 MB/s eta 0:00:00
  Building editable for omnivoice (pyproject.toml) ... done
  Created wheel for omnivoice: filename=omnivoice-0

In [ ]:
from datasets import load_dataset, Audio, concatenate_datasets
import os, json, soundfile as sf
import librosa
import numpy as np

# 1. Configuration for 4-language JURIS dataset
# Waxal Name -> OmniVoice Tag
langs = {
    "yor_tts": "yo",
    "ibo_tts": "ig",
    "hau_tts": "ha",
    "pcm_tts": "pcm"
}
samples_per_gender = 300  # Total 600 per language = 2,400 total
target_sr = 24000

os.makedirs("/content/audio_clean", exist_ok=True)
os.makedirs("/content/data", exist_ok=True)
jsonl_path = "/content/data/train.jsonl"

def is_valid_duration(sample):
    duration = len(sample["audio"]["array"]) / sample["audio"]["sampling_rate"]
    return 2.0 <= duration <= 12.0

all_subsets = []

# 2. Load, Filter, and Label
for subset_name, tag in langs.items():
    print(f"📡 Loading {subset_name}...")
    ds = load_dataset("google/WaxalNLP", subset_name, split="train", trust_remote_code=True)
    ds = ds.cast_column("audio", Audio(sampling_rate=target_sr))

    males = ds.filter(lambda x: x["gender"] == "Male" and is_valid_duration(x))
    females = ds.filter(lambda x: x["gender"] == "Female" and is_valid_duration(x))

    m_sel = males.select(range(min(samples_per_gender, len(males))))
    f_sel = females.select(range(min(samples_per_gender, len(females))))

    # Critical: Attach the correct tag
    m_sel = m_sel.add_column("lang_id", [tag] * len(m_sel))
    f_sel = f_sel.add_column("lang_id", [tag] * len(f_sel))

    all_subsets.extend([m_sel, f_sel])

balanced_dataset = concatenate_datasets(all_subsets).shuffle(seed=42)

# 3. Process Audio & Write JSONL
print(f"🚀 Processing {len(balanced_dataset)} samples...")
with open(jsonl_path, "w") as f:
    for i, sample in enumerate(balanced_dataset):
        audio = sample["audio"]["array"]
        sr = sample["audio"]["sampling_rate"]
        tag = sample["lang_id"]

        audio_trimmed, _ = librosa.effects.trim(audio, top_db=20)
        audio_path = f"/content/audio_clean/{tag}_{i}.wav"
        sf.write(audio_path, audio_trimmed, sr)

        row = {
            "id": f"{tag}_juris_{i}",
            "audio_path": audio_path,
            "text": sample["text"],
            "language_id": tag
        }
        f.write(json.dumps(row) + "\n")

print(f"✅ Multilingual JSONL ready at {jsonl_path}")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'google/WaxalNLP' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'google/WaxalNLP' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


📡 Loading yor_tts...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

data/TTS/yor/yor-train-00000.parquet:   0%|          | 0.00/913M [00:00<?, ?B/s]

... (49 outputs left out)

Filter:   0%|          | 0/1590 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/300 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/300 [00:00<?, ? examples/s]

🚀 Processing 2286 samples...
✅ Multilingual JSONL ready at /content/data/train.jsonl


In [ ]:
import os, glob, json

# Clear old tokens
!rm -rf /content/data/tokens/train
os.makedirs("/content/data/tokens/train/audios", exist_ok=True)
os.makedirs("/content/data/tokens/train/txts", exist_ok=True)

# Extract Tokens
!python -m omnivoice.scripts.extract_audio_tokens \
--input_jsonl /content/data/train.jsonl \
--tar_output_pattern /content/data/tokens/train/audios/shard-%06d.tar \
--jsonl_output_pattern /content/data/tokens/train/txts/shard-%06d.jsonl \
--tokenizer_path eustlb/higgs-audio-v2-tokenizer \
--nj_per_gpu 2 \
--loader_workers 4 \
--samples_per_shard 50 \
--min_num_shards 1 \
--skip_errors \
--shuffle True \
--max_length 12

# 🔥 CRITICAL: Create the data.lst manifest file
jsonl_files = sorted(glob.glob("/content/data/tokens/train/txts/*.jsonl"))
with open("/content/data/tokens/train/data.lst", "w") as f:
    for file_path in jsonl_files:
        f.write(file_path + "\n")

# Create data_config.json
data_config = {"train": [{"manifest_path": ["/content/data/tokens/train/data.lst"]}]}
with open("/content/data_config.json", "w") as f:
    json.dump(data_config, f, indent=4)

print(f"✅ Created data.lst and data_config for {len(jsonl_files)} shards.")

2026-04-20 22:13:24,813 INFO [extract_audio_tokens.py:341] Input mode: raw JSONL (/content/data/train.jsonl)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
2026-04-20 22:13:24,835 INFO [extract_audio_tokens.py:427] GPU count: 1, Processes per GPU: 2, Total processes: 2
Extracting Audio Tokens:   0% 0/2286 [00:00<?, ?it/s]2026-04-20 22:13:31,814 INFO [extract_audio_tokens.py:548] Submitting tasks... (2 workers)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our sugg

In [ ]:
import json
import os

# Environment setup
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["TRANSFORMERS_NO_FLEX_ATTENTION"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

train_config = {
    "llm_name_or_path": "Qwen/Qwen2.5-0.5B", # Updated for latest OmniVoice base
    "audio_vocab_size": 1025,
    "audio_mask_id": 1024,
    "num_audio_codebook": 8,
    "audio_codebook_weights": [8, 8, 6, 6, 4, 4, 2, 2],
    "init_from_checkpoint": "k2-fsa/OmniVoice",

    "learning_rate": 4e-5,
    "steps": 1200,          # Increased for 4-language diversity
    "batch_tokens": 1024,
    "gradient_accumulation_steps": 2, # Helps stability on T4
    "num_workers": 1,
    "mixed_precision": "no",
    "max_grad_norm": 1.0,
    "warmup_ratio": 0.1,    # 10% warmup for smoother convergence
    "logging_steps": 10,
    "save_steps": 300,
    "attn_implementation": "eager"
}

with open("/content/train_config.json", "w") as f:
    json.dump(train_config, f, indent=4)

print("✅ Multilingual JURIS train_config ready.")

✅ Multilingual JURIS train_config ready.


In [ ]:
import json
import glob
import os

# Paths
audio_shards = sorted(glob.glob("/content/data/tokens/train/audios/*.tar"))
jsonl_shards = sorted(glob.glob("/content/data/tokens/train/txts/*.jsonl"))
manifest_path = "/content/data/tokens/train/data.lst"

# Check if counts match
if len(audio_shards) != len(jsonl_shards):
    print(f"⚠️ Warning: Found {len(audio_shards)} tar files and {len(jsonl_shards)} jsonl files.")

print("🔨 Generating WebDataset manifest...")

with open(manifest_path, "w") as manifest:
    for tar, jsl in zip(audio_shards, jsonl_shards):
        num_items = 0
        total_seconds = 0.0

        # Read the jsonl to get stats for this shard
        with open(jsl, "r") as f:
            for line in f:
                data = json.loads(line)
                num_items += 1
                # The duration was stored/calculated during extraction
                # If not present, we estimate or use your max_length
                total_seconds += data.get("duration", 10.0)

        # Format: tar_path label_jsonl_path num_items num_seconds
        manifest_line = f"{tar} {jsl} {num_items} {total_seconds}\n"
        manifest.write(manifest_line)

print(f"✅ Success! Compliant manifest created at {manifest_path}")

🔨 Generating WebDataset manifest...
✅ Success! Compliant manifest created at /content/data/tokens/train/data.lst


In [ ]:
# Launching the JURIS v2 training run
!accelerate launch \
    --num_processes 1 \
    --mixed_precision no \
    -m omnivoice.cli.train \
    --train_config /content/train_config.json \
    --data_config /content/data_config.json \
    --output_dir /content/drive/MyDrive/JURIS_Omnivoice_v2

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
Fetching 13 files: 100% 13/13 [00:00<00:00, 54855.08it/s]
Download complete: : 0.00B [00:00, ?B/s]
Fetching 13 files: 100% 13/13 [00:00<00:00, 19564.39it/s]
Download complete: : 0.00B [00:00, ?B/s]
Loading weights: 100% 313/313 [00:00<00:00, 4938.73it/s]
04/20/2026 22:24:56 - INFO - omnivoice.training.trainer - Loaded Config: TrainingConfig(output_dir='/content/drive/MyDrive/JURIS_Omnivoice_v2', data_config='/content/data_config.json', llm_name_or_path='Qwen/Qwen2.5-0.5B', audio_vocab_size=1025, audio_mask_id=1024, num_audio_codebook=8, audio_codebook_weights=[8, 8, 6, 6, 4, 4, 2, 2], drop_cond_ratio=0.1, prompt_ratio_range=(0.0, 0.3), mask_ratio_range=(0.0, 1.0), language_ratio=0.8, use_pinyin_ratio=0.

In [ ]:
import os
from google.colab import drive

# 1. Mount Drive if you haven't already
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. Set your paths
source_folder = "/content/drive/MyDrive/JURIS_Omnivoice_v2"
zip_filename = "/content/JURIS_Omnivoice_v2_Full.zip"
destination_drive_path = "/content/drive/MyDrive/JURIS_Final_Models/"

# 3. Create the destination folder in Drive
!mkdir -p "{destination_drive_path}"

# 4. Zip the training output (Fast & preserves folder structure)
print("📦 Zipping model files... this may take a minute.")
!zip -r {zip_filename} {source_folder}

# 5. Move the Zip to your Drive
print("🚀 Moving zip to Google Drive...")
!cp {zip_filename} "{destination_drive_path}"

# 6. Final Sync check
drive.flush_and_unmount()
print(f"✅ Success! Your model is safe in: {destination_drive_path}")

📦 Zipping model files... this may take a minute.
  adding: content/drive/MyDrive/JURIS_Omnivoice_v2/ (stored 0%)
  adding: content/drive/MyDrive/JURIS_Omnivoice_v2/train.log (deflated 83%)
  adding: content/drive/MyDrive/JURIS_Omnivoice_v2/checkpoint-600/ (stored 0%)
  adding: content/drive/MyDrive/JURIS_Omnivoice_v2/checkpoint-600/optimizer.bin (deflated 31%)
  adding: content/drive/MyDrive/JURIS_Omnivoice_v2/checkpoint-600/scheduler.bin (deflated 62%)
  adding: content/drive/MyDrive/JURIS_Omnivoice_v2/checkpoint-600/model.safetensors (deflated 9%)
  adding: content/drive/MyDrive/JURIS_Omnivoice_v2/checkpoint-600/tokenizer.json (deflated 81%)
  adding: content/drive/MyDrive/JURIS_Omnivoice_v2/checkpoint-600/config.json (deflated 71%)
  adding: content/drive/MyDrive/JURIS_Omnivoice_v2/checkpoint-600/chat_template.jinja (deflated 76%)
  adding: content/drive/MyDrive/JURIS_Omnivoice_v2/checkpoint-600/tokenizer_config.json (deflated 53%)
  adding: content/drive/MyDrive/JURIS_Omnivoice_v2/

In [ ]:
import os
import shutil

# List of folders that are safe to delete after training is complete
folders_to_clear = [
    "/content/audio_clean",      # Raw processed audio
    "/content/data/tokens",       # Training shards
    "/root/.cache/huggingface"   # Cached datasets (can be re-downloaded if needed)
]

for folder in folders_to_clear:
    if os.path.exists(folder):
        print(f"🧹 Deleting {folder}...")
        shutil.rmtree(folder)
    else:
        print(f"✅ {folder} already empty.")

print("\n🚀 Disk space recovered! Your checkpoints remain untouched.")

🧹 Deleting /content/audio_clean...
🧹 Deleting /content/data/tokens...
🧹 Deleting /root/.cache/huggingface...

🚀 Disk space recovered! Your checkpoints remain untouched.


In [ ]:
import torch
import soundfile as sf
from omnivoice import OmniVoice
from IPython.display import Audio, display

# 1. LOAD YOUR NEW MODEL
model_path = "/content/drive/MyDrive/JURIS_Omnivoice_v2/checkpoint-1200"
model = OmniVoice.from_pretrained(model_path).to("cuda")

# 2. DEFINE THE TEST CASES
test_cases = [
    {"lang": "yo", "text": "Ẹ lẹ́ẹ̀ o, èmi ni MTN Zigi. Mo lè ràn yín lọ́wọ́ lórí ọ̀rọ̀ bọ́ńdù àti iye owó ìpè yín. Ẹ bèèrè lọ́wọ́ mi ní ohunkóhun."},
    {"lang": "ig", "text": "Ndéèwó, mụ bụ MTN Zigi. Nwere m ike inyere gị aka maka ajụjụ gbasara bọndụ gị na airtime gị. Jụọ m ihe ọbụla."},
    {"lang": "pcm", "text": "Hello, I be MTN Zigi. I fit help you check your bundle and airtime mata. Abeg, feel free to ask me anything."},
    {"lang": "en", "text": "Hello, I am MTN Zigi. I can assist you with inquiries as regards your bundle and airtime. Feel free to ask me anything."}
]

# 3. RUN INFERENCE
print("🎙️ Generating samples with JURIS v2...")
with torch.inference_mode():
    for test in test_cases:
        print(f"\nTesting Language: {test['lang'].upper()}")
        audio = model.generate(
            text=test['text'],
            language=test['lang'],
            num_step=20,
            guidance_scale=5.0
        )

        filename = f"test_{test['lang']}.wav"
        sf.write(filename, audio[0], 24000)
        display(Audio(filename, rate=24000))

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

🎙️ Generating samples with JURIS v2...

Testing Language: YO



Testing Language: IG



Testing Language: PCM



Testing Language: EN


In [ ]:
import torch
import soundfile as sf
from omnivoice import OmniVoice
from IPython.display import Audio, display

# 1. LOAD YOUR JURIS MODEL
model_path = "/content/drive/MyDrive/JURIS_Omnivoice_v2/checkpoint-1200"
model = OmniVoice.from_pretrained(model_path).to("cuda")

# 2. THE MULTILINGUAL ZIGI SUITE
# Grouping the transcriptions by language tag
test_suite = [
    {
        "lang": "en",
        "label": "English",
        "text": "Hello, I am MTN Zigi. I can assist you with inquiries as regards your bundle and airtime. Feel free to ask me anything."
    },
    {
        "lang": "yo",
        "label": "Yoruba",
        "text": "Ẹ lẹ́ẹ̀ o, èmi ni MTN Zigi. Mo lè ràn yín lọ́wọ́ lórí ọ̀rọ̀ bọ́ńdù àti iye owó ìpè yín. Ẹ bèèrè lọ́wọ́ mi ní ohunkóhun."
    },
    {
        "lang": "ig",
        "label": "Igbo",
        "text": "Ndéèwó, mụ bụ MTN Zigi. Nwere m ike inyere gị aka maka ajụjụ gbasara bọndụ gị na airtime gị. Jụọ m ihe ọbụla."
    },
    {
        "lang": "ha",
        "label": "Hausa",
        "text": "Barka dai, ni ne MTN Zigi. Zan iya taimaka maka da bayani game da kundin tsarinka da katin magana. Tambaye ni duk abin da kake so."
    },
    {
        "lang": "pcm",
        "label": "Pidgin",
        "text": "Hello, I be MTN Zigi. I fit help you check your bundle and airtime mata. Abeg, feel free to ask me anything."
    }
]

# 3. GENERATE ALL SAMPLES
print("🎙️ Launching Multilingual Inference for JURIS v2...")
print("-" * 50)

with torch.inference_mode():
    for test in test_suite:
        print(f"\n🔊 Generating {test['label']} ({test['lang']})...")

        audio = model.generate(
            text=test['text'],
            language=test['lang'],
            num_step=20,
            guidance_scale=5.0,  # High scale to enforce the fine-tuned accent
            speed=1.0            # Standard professional speed
        )

        filename = f"zigi_{test['lang']}_final.wav"
        sf.write(filename, audio[0], 24000)

        print(f"Transcription: {test['text']}")
        display(Audio(filename, rate=24000))
        print("-" * 30)

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

🎙️ Launching Multilingual Inference for JURIS v2...
--------------------------------------------------

🔊 Generating English (en)...
Transcription: Hello, I am MTN Zigi. I can assist you with inquiries as regards your bundle and airtime. Feel free to ask me anything.


------------------------------

🔊 Generating Yoruba (yo)...
Transcription: Ẹ lẹ́ẹ̀ o, èmi ni MTN Zigi. Mo lè ràn yín lọ́wọ́ lórí ọ̀rọ̀ bọ́ńdù àti iye owó ìpè yín. Ẹ bèèrè lọ́wọ́ mi ní ohunkóhun.


------------------------------

🔊 Generating Igbo (ig)...
Transcription: Ndéèwó, mụ bụ MTN Zigi. Nwere m ike inyere gị aka maka ajụjụ gbasara bọndụ gị na airtime gị. Jụọ m ihe ọbụla.


------------------------------

🔊 Generating Hausa (ha)...
Transcription: Barka dai, ni ne MTN Zigi. Zan iya taimaka maka da bayani game da kundin tsarinka da katin magana. Tambaye ni duk abin da kake so.


------------------------------

🔊 Generating Pidgin (pcm)...
Transcription: Hello, I be MTN Zigi. I fit help you check your bundle and airtime mata. Abeg, feel free to ask me anything.


------------------------------


In [ ]:
import torch
import os
import soundfile as sf
from omnivoice import OmniVoice
from google.colab import drive
from IPython.display import Audio

# 1. Ensure Drive is mounted
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. Point to the EXACT folder in your Drive
# This is the path we saved to earlier
drive_model_path = "/content/drive/MyDrive/JURIS_Omnivoice_v2/checkpoint-1200"

print(f"📡 Attempting to load model from: {drive_model_path}")

try:
    # 3. Load from Drive
    model = OmniVoice.from_pretrained(drive_model_path).to("cuda")
    print("✅ SUCCESS: Model loaded directly from Google Drive!")

    # 4. Quick "Proof of Life" Inference (Pidgin)
    test_text = "No shaking, the model de for Google Drive correct-correct."

    with torch.inference_mode():
        audio = model.generate(
            text=test_text,
            language="pcm",
            num_step=20,
            guidance_scale=5.0
        )

    sf.write("drive_test_output.wav", audio[0], 24000)
    display(Audio("drive_test_output.wav", rate=24000))

except Exception as e:
    print(f"❌ ERROR: Could not find model on Drive. Detailed error: {e}")

📡 Attempting to load model from: /content/drive/MyDrive/JURIS_Omnivoice_v2/checkpoint-1200


Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

✅ SUCCESS: Model loaded directly from Google Drive!


In [ ]:
import torch
import soundfile as sf
from omnivoice import OmniVoice
from IPython.display import Audio, display
import os

# 1. LOAD MODEL FROM DRIVE
model_path = "/content/drive/MyDrive/JURIS_Omnivoice_v2/checkpoint-1200"
if not os.path.exists(model_path):
    print("❌ Model path not found. Check Drive mount.")
else:
    model = OmniVoice.from_pretrained(model_path).to("cuda")

# 2. THE REFINED SUITE
test_suite = [
    {
        "lang": "en",
        "label": "English",
        "text": "Good day, I am MTN Zigi. I can assist you with inquiries as regards your bundle and airtime. Feel free to ask me anything. We have exciting news for you and we promise to serve you better."
    },
    {
        "lang": "yo",
        "label": "Yoruba",
        # REPLACED: 'Ẹ lẹ́ẹ̀ o' with 'Ẹ kú àkókò yìí o'
        "text": "Ẹ kú àkókò yìí o, èmi ni MTN Zigi. Mo lè ràn yín lọ́wọ́ lórí ọ̀rọ̀ bọ́ńdù àti iye owó ìpè yín. Ẹ bèèrè lọ́wọ́ mi ní ohunkóhun. A ní ìròyìn ayọ̀ fún yín, a sì ṣe ìlérí láti sìn yín dáadáa ju tẹ́lẹ̀ lọ."
    },
    {
        "lang": "ig",
        "label": "Igbo",
        "text": "Ndéèwó, mụ bụ MTN Zigi. Nwere m ike inyere gị aka maka ajụjụ gbasara bọndụ gị na airtime gị. Jụọ m ihe ọbụla. Anyị nwere ozi na-atọ ụtọ maka gị, anyị na-ekwekwa ná nkwa ịrụrụ gị ọrụ nke ọma karịa."
    },
    {
        "lang": "ha",
        "label": "Hausa",
        "text": "Barka dai, ni ne MTN Zigi. Zan iya taimaka maka da bayani game da kundin tsarinka da katin magana. Tambaye ni duk abin da kake so. Muna da labarai masu kayatarwa a gare ku, kuma mun yi muku alkawarin yi muku hidima mafi kyau."
    },
    {
        "lang": "pcm",
        "label": "Pidgin",
        "text": "Good day, I be MTN Zigi. I fit help you check your bundle and airtime mata. Abeg, feel free to ask me anything. We get beta news for you, and we promise say we go serve you well-well pass before."
    }
]

# 3. GENERATE ALL SAMPLES
print("🎙️ Generating Refined Zigi Samples...")

with torch.inference_mode():
    for test in test_suite:
        print(f"\n🔊 {test['label']} ({test['lang']})")
        audio = model.generate(
            text=test['text'],
            language=test['lang'],
            num_step=20,
            guidance_scale=5.0
        )
        filename = f"zigi_refined_{test['lang']}.wav"
        sf.write(filename, audio[0], 24000)
        display(Audio(filename, rate=24000))

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

🎙️ Generating Refined Zigi Samples...

🔊 English (en)



🔊 Yoruba (yo)



🔊 Igbo (ig)



🔊 Hausa (ha)



🔊 Pidgin (pcm)


In [ ]:
import torch
import os
import soundfile as sf
from omnivoice import OmniVoice
from google.colab import drive
from IPython.display import Audio

# 1. Ensure Drive is mounted
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. Point to the EXACT folder in your Drive
# This is the path we saved to earlier
drive_model_path = "/content/drive/MyDrive/JURIS_Omnivoice_v2/checkpoint-1200"

print(f"📡 Attempting to load model from: {drive_model_path}")

try:
    # 3. Load from Drive
    model = OmniVoice.from_pretrained(drive_model_path).to("cuda")
    print("✅ SUCCESS: Model loaded directly from Google Drive!")

    # 4. Quick "Proof of Life" Inference (Pidgin)
    test_text = "No shaking, the model de for Google Drive correct-correct."

    with torch.inference_mode():
        audio = model.generate(
            text=test_text,
            language="pcm",
            num_step=20,
            guidance_scale=5.0
        )

    sf.write("drive_test_output.wav", audio[0], 24000)
    display(Audio("drive_test_output.wav", rate=24000))

except Exception as e:
    print(f"❌ ERROR: Could not find model on Drive. Detailed error: {e}")

In [ ]:
import torch
import soundfile as sf
from omnivoice import OmniVoice
from IPython.display import Audio, display
import os

# 1. LOAD MODEL FROM DRIVE
model_path = "/content/drive/MyDrive/JURIS_Omnivoice_v2/checkpoint-1200"
if not os.path.exists(model_path):
    print("❌ Model path not found. Check Drive mount.")
else:
    model = OmniVoice.from_pretrained(model_path).to("cuda")

# 2. THE REFINED SUITE
test_suite = [
    {
        "lang": "en",
        "label": "English",
        "text": "Good day, I am MTN Zigi. I can assist you with inquiries as regards your bundle and airtime. Feel free to ask me anything. We have exciting news for you and we promise to serve you better."
    },
    {
        "lang": "yo",
        "label": "Yoruba",
        # REPLACED: 'Ẹ lẹ́ẹ̀ o' with 'Ẹ kú àkókò yìí o'
        "text": "Ẹ káàbọ̀, èmi ni MTN Zigi. Mo lè ràn ọ́ lọ́wọ́ pẹ̀lú àwọn ìbéèrè tó ní í ṣe pẹ̀lú bundle àti airtime yín. Ẹ má ṣe ṣiyèméjì láti bi mí ohunkóhun. A ní ìròyìn alárinrin fún yín, a sì ṣe ìlérí láti máa sin yín dáadáa ju tẹ́lẹ̀ lọ."
    },
    {
        "lang": "ig",
        "label": "Igbo",
        "text": "Ndeewo, abụ m MTN Zigi. Enwere m ike inyere gị aka na ajụjụ gbasara bundle na airtime gị. Biko nwee onwe gị ịjụ m ihe ọ bụla. Anyị nwere ozi ọma maka gị, ma anyị na-ekwe nkwa ịrụ gị ọrụ nke ọma karịa ka ọ dị na mbụ."
    },
    {
        "lang": "ha",
        "label": "Hausa",
        "text": "Sannu, ni ne MTN Zigi. Zan iya taimaka maka tambayoyinku dangane da bundle da airtime ɗinku. Kada ku yi shakka ku tambaye ni komai. Muna da labari mai daɗi a gare ku, kuma muna alƙawarin yi muku hidima fiye da da."
    },
    {
        "lang": "pcm",
        "label": "Pidgin",
        "text": "Good day, I be MTN Zigi. I fit help you check your bundle and airtime mata. Abeg, feel free to ask me anything. We get beta news for you, and we promise say we go serve you well-well pass before."
    }
]

# 3. GENERATE ALL SAMPLES
print("🎙️ Generating Refined Zigi Samples...")

with torch.inference_mode():
    for test in test_suite:
        print(f"\n🔊 {test['label']} ({test['lang']})")
        audio = model.generate(
            text=test['text'],
            language=test['lang'],
            num_step=20,
            guidance_scale=5.0
        )
        filename = f"zigi_refined_{test['lang']}.wav"
        sf.write(filename, audio[0], 24000)
        display(Audio(filename, rate=24000))

❌ Model path not found. Check Drive mount.
🎙️ Generating Refined Zigi Samples...

🔊 English (en)


/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


NameError: name 'model' is not defined

In [ ]:
import os
from google.colab import drive

# 1. Mount Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. Search for common fine-tuning file patterns
print("🔍 Searching for your JURIS weights...")
found_paths = []

# We are looking for 'model.safetensors' or 'optimizer.pt' which are the heavy model files
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    if 'checkpoint-1200' in root or 'model.safetensors' in files:
        found_paths.append(root)

if found_paths:
    print("\n✅ Found your model in these locations:")
    for path in found_paths:
        print(f"📍 {path}")
else:
    print("\n❌ No model files found in Drive. They might have stayed in Colab's local disk.")

Mounted at /content/drive
🔍 Searching for your JURIS weights...

❌ No model files found in Drive. They might have stayed in Colab's local disk.
